# YMG3 – CNN Ödevi: Fashion-MNIST

Bu defter, ödev belgesindeki 3–8. bölümlerin kod uygulamalarını sırayla içerir. Google Colab'da **Çalışma zamanı → Tümünü çalıştır** ile baştan sona çalıştırın.

> Not: Eğitim doğrulukları, test doğruluğu ve süreler çalıştırma sırasında hesaplanır; önceden uydurulmuş değer kullanılmaz.

## 3. Bölüm – Konvolüsyon işleminin kodlanması

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time

tf.keras.utils.set_random_seed(42)

conv = tf.keras.layers.Conv2D(
    filters=16,
    kernel_size=(3, 3),
    activation='relu',
    kernel_initializer='he_normal'
)

dummy_input = tf.random.normal((1, 128, 128, 1))
output = conv(dummy_input)
weights, biases = conv.get_weights()

print("Giriş boyutu:", dummy_input.shape)
print("Çıkış boyutu:", output.shape)
print("Filtre ağırlıklarının boyutu:", weights.shape)
print("Bias boyutu:", biases.shape)
print("İlk filtre:")
print(weights[:, :, 0, 0])

### 16 filtrenin başlangıç ağırlıklarını görselleştirme

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(weights[:, :, 0, i], cmap='gray')
    plt.title(f'Filtre {i + 1}')
    plt.axis('off')
plt.suptitle("16 Konvolüsyon Filtresinin Başlangıç Ağırlıkları")
plt.tight_layout()
plt.show()

## 4. Bölüm – Pooling işleminin incelenmesi

In [ ]:
input_data = tf.constant([
    [[1., 3., 2., 4.],
     [5., 6., 1., 2.],
     [7., 2., 8., 3.],
     [4., 1., 5., 9.]]
])
input_data = tf.reshape(input_data, (1, 4, 4, 1))

pool = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2)
pool_output = pool(input_data)

print("Giriş:")
print(input_data.numpy()[0, :, :, 0])
print("Pooling sonucu:")
print(pool_output.numpy()[0, :, :, 0])
print("Giriş boyutu:", input_data.shape)
print("Çıkış boyutu:", pool_output.shape)

## 5. Bölüm – ReLU ve Softmax

In [ ]:
x = tf.constant([-5., -2., -1., 0., 2., 4., 6.])
relu = tf.keras.layers.ReLU()
y = relu(x)
print("Giriş:", x.numpy())
print("ReLU çıkışı:", y.numpy())

In [ ]:
logits = tf.constant([[2.0, 1.0, 0.1]])
softmax = tf.keras.layers.Softmax()
probabilities = softmax(logits)
print("Softmax sonuçları:")
print(probabilities.numpy())
print("Olasılıkların toplamı:")
print(tf.reduce_sum(probabilities).numpy())
print("En yüksek olasılığın sınıf indeksi:", tf.argmax(probabilities, axis=1).numpy())

## 6. Bölüm A – Fashion-MNIST veri setinin yüklenmesi

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("Eğitim verisi:", x_train.shape)
print("Test verisi:", x_test.shape)

## 6. Bölüm B – CNN mimarisinin oluşturulması

In [ ]:
def model_olustur(ilk_filtre_sayisi=8):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(ilk_filtre_sayisi, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    return model

model = model_olustur(8)
model.summary()

## 6. Bölüm C – Modelin derlenmesi ve eğitilmesi

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

## 6. Bölüm D – Test başarısının ölçülmesi

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Başarısı: %{:.2f}".format(test_accuracy * 100))

## 6. Bölüm E – Eğitim ve doğrulama grafiklerinin çizilmesi

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 7. Bölüm – Mimari boyutların kontrolü

Beklenen boyutlar: 28×28×1 → 26×26×8 → 13×13×8 → 11×11×16 → 5×5×16. Flatten çıktısı 5×5×16=400 elemandır.

In [ ]:
# Katman çıkış boyutlarını otomatik inceleme
for layer in model.layers:
    print(f"{layer.name}: {layer.output.shape}")

## 8. Bölüm – 8 ve 16 filtreli modellerin karşılaştırılması

Aşağıdaki hücre iki modeli aynı epoch ve batch ayarlarıyla eğitir. İlk konvolüsyon katmanındaki filtre sayısı dışında mimari ve eğitim parametreleri aynıdır. Eğitim süresi, eğitim doğruluğu ve test doğruluğu ekrana yazdırılır.

In [ ]:
def deney_calistir(filtre_sayisi):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    deney_modeli = model_olustur(filtre_sayisi)
    deney_modeli.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    baslangic = time.perf_counter()
    deney_history = deney_modeli.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )
    egitim_suresi = time.perf_counter() - baslangic
    test_loss, test_accuracy = deney_modeli.evaluate(x_test, y_test, verbose=0)

    return {
        "filtre": filtre_sayisi,
        "egitim_dogrulugu": deney_history.history["accuracy"][-1],
        "test_dogrulugu": test_accuracy,
        "egitim_suresi_sn": egitim_suresi
    }

sonuc_8 = deney_calistir(8)
sonuc_16 = deney_calistir(16)

print("\nDENEY KARŞILAŞTIRMASI")
print(f"{'Ölçüt':<25}{'8 filtre':>15}{'16 filtre':>15}")
print("-" * 55)
print(f"{'Eğitim doğruluğu':<25}{sonuc_8['egitim_dogrulugu']:>15.4f}{sonuc_16['egitim_dogrulugu']:>15.4f}")
print(f"{'Test doğruluğu':<25}{sonuc_8['test_dogrulugu']:>15.4f}{sonuc_16['test_dogrulugu']:>15.4f}")
print(f"{'Eğitim süresi (sn)':<25}{sonuc_8['egitim_suresi_sn']:>15.2f}{sonuc_16['egitim_suresi_sn']:>15.2f}")

## 9. Bölüm – Sonuçların raporlanması

Yukarıdaki deney hücresinin ürettiği gerçek değerleri rapordaki karşılaştırma tablosuna aktarın. Filtre sayısının artmasının başarıyı her zaman artırmadığını; model kapasitesi ve hesaplama süresi üzerindeki etkileriyle birlikte değerlendirin.

**Teslim öncesi:** Tüm hücreleri çalıştırın, grafik ve çıktı ekran görüntülerini saklayın. Google Colab'da Dosya → İndir → .ipynb seçeneğiyle defteri indirebilirsiniz.